# CL-MEE — Reproduction and reviewer-revision experiments

This notebook is a front-end for `run_revision_experiments.py`. The default run loads the supplied validation-trained scorer checkpoint (`foursquare_model_weights.json`) so the reference ranking is not affected by small platform-specific optimizer differences. Use `--refit-weights` in the script for an optional optimizer refit.

**Run from this folder** with `dataset_TSMC2014_NYC.txt` present. No network access or external data download is required.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
DATASET = ROOT / "dataset_TSMC2014_NYC.txt"
OUTPUT = ROOT / "notebook_run_outputs"
assert DATASET.exists(), f"Dataset not found: {DATASET}"


## 1. Run the unit tests

The tests check the display-budget invariant, zero-evidence handling, ranking preservation in the post-hoc baseline, the common-anchor calculation, score-envelope compliance, and the explicit identity control.


In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
               cwd=ROOT, check=True)


## 2. Re-run all experiments

The complete run includes 10 simulation seeds (24,000 sessions) and 20,223 eligible Foursquare test transitions. The script saves machine-readable JSON and CSV tables under `revision_outputs/`.


In [ ]:
subprocess.run([sys.executable, "run_revision_experiments.py",
                "--dataset", str(DATASET),
                "--output-dir", str(OUTPUT)],
               cwd=ROOT, check=True)


## 3. Inspect the principal results


In [ ]:
results = json.loads((OUTPUT / "revision_results.json").read_text(encoding="utf-8"))
simulation_table = pd.DataFrame(results["simulation"]["table"])
foursquare_table = pd.DataFrame(results["foursquare"]["main_comparison_table"])
ablations = pd.DataFrame(results["foursquare"]["ablation_table"])
display(simulation_table)
display(foursquare_table)
display(ablations)


## 4. Operational diagnostics and reproducibility


In [ ]:
f = results["foursquare"]
print("Protocol:")
print(json.dumps(f["protocol"], indent=2))
print("\nOperational diagnostics:")
for policy, values in f["operational_diagnostics"].items():
    print(policy, json.dumps({k: values[k] for k in [
        "B", "lambda", "epsilon", "NDCG_retained_pct", "rerank_rate",
        "pre_truncation_overbudget_rate_positive_D",
        "coverage_below_rho_rate_positive_D", "score_loss_all",
        "score_loss_promoted", "promotion_depth_promoted",
        "score_loss_exceeds_epsilon_count"
    ]}, indent=2))
print("\nAnchor sensitivity:")
print(json.dumps(f["comparison_anchor_sensitivity"], indent=2))
print("\nRuntime:")
print(json.dumps(f["runtime"], indent=2))
print("\nObserved positive/negative contribution example:")
print(json.dumps(f["positive_negative_example"], indent=2))


## 5. Output files and final verification

- `revision_outputs/revision_results.json`: reference results and metadata.
- CSV exports: simulation, Foursquare main comparison, ablations, and signed-contribution example.
- `run_manifest.json`: dataset hash, checkpoint hash, and environment.
- `verify_package.py`: verifies JSON/CSV consistency, dataset/checkpoint hashes, invariants, and the corrected paper metrics.
- `capture_environment.py`: captures platform and `pip freeze`.

The original submitted notebook is retained separately as `clmee_original_notebook_legacy.ipynb` for audit.

## 6. Verify the generated run against the reference manuscript

This checks the generated JSON/CSV exports, dataset and checkpoint hashes, budget invariants, and the corrected P1/P2 values in `main.tex`.

In [ ]:
subprocess.run([sys.executable, "verify_package.py",
                "--results-dir", str(OUTPUT),
                "--dataset", str(DATASET),
                "--paper", str(ROOT.parent / "main.tex"),
                "--reference-dir", str(ROOT / "revision_outputs")],
               cwd=ROOT, check=True)
